In [5]:
## Import required libraries
from dotenv import load_dotenv
from typing import Annotated
from pydantic import BaseModel
from pathlib import Path
from datetime import datetime
import re

## LangGraph and LangChain
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode, tools_condition
from langchain_openai import ChatOpenAI
from langchain_core.tools import Tool
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage
from langchain_community.utilities import GoogleSerperAPIWrapper

# Load environment variables
load_dotenv(override=True)

True

In [20]:
## Load topics from Final1.txt
def load_topics(file_path: str = "output/Final1.txt"):
    """Load topics from the Final1.txt file."""
    topics = []
    with open(file_path, 'r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if line and not line.isspace():
                # Remove leading numbers and dots (e.g., "1.U.K. PM..." -> "U.K. PM...")
                topic = re.sub(r'^\d+\.', '', line).strip()
                if topic:
                    topics.append(topic)
    return topics

# Load all topics
topics = load_topics()
print(f"Loaded {len(topics)} topics:")
for i, topic in enumerate(topics, 1):
    print(f"{i}. {topic}")

Loaded 16 topics:
1. U.K. PM Starmer, Trump discuss security and Russia deterrence in Arctic region
2. Internet shutdown in Iran as protests mount; Trump warns of U.S. action
3. Russian strikes leave one million without water, heat in Ukraine
4. U.S. Senate rejects Trump on Venezuela in war powers vote
5. Denmark, Greenland envoys meet White House officials over Trump's call for 'takeover'
6. Dhaka suspends visa service in India, seeks U.S. visa bond exemption
7. U.K. supported U.S. mission to seize Russian-flagged Bella-1 in the North Atlantica87.India plans to scrap curbs on Chinese firms bidding for government contracts
8. Three Indians among crew on board Russian oil tanker seized by U.S.: report
9. Air India takes delivery of first Dreamliner after privatisation
10. PM Modi chairs roundtable with AI startups ahead of February summit
11. Reliance Industries would consider buying Venezuelan oil if allowed
12. Robot dog pulls sleds in China
13. NASA administrator orders early return 

In [21]:
## Setup LLM and Search Tool

# Initialize Google Serper API search tool
serper = GoogleSerperAPIWrapper()

# Create a search tool that limits results (max 1 search per topic)
def limited_search(query: str) -> str:
    """Perform a web search with limited results (max 1 search)."""
    results = serper.run(query)
    # If results are long, truncate to first part
    # Limit to 2000 chars to keep it focused
    return results[:2000] if len(results) > 2000 else results

# Wrap in LangChain Tool
search_tool = Tool(
    name="web_search",
    func=limited_search,
    description="Search the web for information about a topic. Use this to find current information about news topics. Search once per topic."
)

tools = [search_tool]

# Initialize LLM
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.7)
llm_with_tools = llm.bind_tools(tools)

print("LLM and Search Tool initialized successfully!")

LLM and Search Tool initialized successfully!


In [22]:
## Define State for LangGraph

class State(BaseModel):
    messages: Annotated[list, add_messages]

In [23]:
## Create LangGraph workflow

def create_news_graph():
    """Create a LangGraph for processing news topics."""
    graph_builder = StateGraph(State)
    
    def llm_node(state: State) -> State:
        """LLM node that processes messages and can call tools."""
        response = llm_with_tools.invoke(state.messages)
        return State(messages=[response])
    
    graph_builder.add_node("llm_node", llm_node)
    graph_builder.add_node("tools", ToolNode(tools=tools))
    
    # Add edges - tools_condition routes to "tools" if tool calls exist, otherwise END
    graph_builder.add_edge(START, "llm_node")
    graph_builder.add_conditional_edges(
        "llm_node",
        tools_condition,
        {
            "tools": "tools",
            "__end__": END
        }
    )
    # After tools execute, loop back to llm_node (which will use the same conditional edge)
    graph_builder.add_edge("tools", "llm_node")
    
    return graph_builder.compile()

# Create the graph
graph = create_news_graph()
print("LangGraph created successfully!")

LangGraph created successfully!


In [24]:
## Process each topic and generate kid-friendly explanations

# System message for kid-friendly explanations
KID_FRIENDLY_SYSTEM_MESSAGE = """You are a friendly and fun news explainer for kids! Your job is to explain news topics in a cool, exciting, and easy-to-understand way that kids will love.

Format your response as a fun, friendly explanation that a kid would enjoy reading."""

all_explanations = []

print("Starting to process topics...\n")
print("=" * 80)

for i, topic in enumerate(topics, 1):
    print(f"\nProcessing topic {i}/{len(topics)}: {topic}")
    
    # Create messages for this topic
    messages = [
        SystemMessage(content=KID_FRIENDLY_SYSTEM_MESSAGE),
        HumanMessage(content=f"""Explain the below topic in cool and kids friendly way - search the internet(search once only) and get the correct information before summarizing it.

{topic}

Make sure to highlight when it is happened.""")
    ]
    
    # Run the graph - pass State object, but result will be a dict
    initial_state = State(messages=messages)
    try:
        final_state = graph.invoke(initial_state)
        
        # Extract the explanation from the final messages (get the last AI message with content)
        # Note: graph.invoke returns a dict, not a State object
        explanation = ""
        messages_list = final_state.get("messages", [])
        for msg in reversed(messages_list):
            if isinstance(msg, AIMessage):
                if hasattr(msg, 'content') and msg.content and msg.content.strip():
                    explanation = msg.content
                    break  # Get the last meaningful AI response
        
        if explanation:
            explanation_text = f"**Topic {i}: {topic}**\n\n{explanation}\n\n"
            all_explanations.append(explanation_text)
            print(f"✓ Successfully processed topic {i}")
        else:
            print(f"⚠ No explanation generated for topic {i}")
            
    except Exception as e:
        print(f"✗ Error processing topic {i}: {str(e)}")
        error_text = f"**Topic {i}: {topic}**\n\n[Error: Could not process this topic]\n\n"
        all_explanations.append(error_text)

print("\n" + "=" * 80)
print(f"\nCompleted processing {len(topics)} topics!")

Starting to process topics...


Processing topic 1/16: U.K. PM Starmer, Trump discuss security and Russia deterrence in Arctic region
✓ Successfully processed topic 1

Processing topic 2/16: Internet shutdown in Iran as protests mount; Trump warns of U.S. action
✓ Successfully processed topic 2

Processing topic 3/16: Russian strikes leave one million without water, heat in Ukraine
✓ Successfully processed topic 3

Processing topic 4/16: U.S. Senate rejects Trump on Venezuela in war powers vote
✓ Successfully processed topic 4

Processing topic 5/16: Denmark, Greenland envoys meet White House officials over Trump's call for 'takeover'
✓ Successfully processed topic 5

Processing topic 6/16: Dhaka suspends visa service in India, seeks U.S. visa bond exemption
✓ Successfully processed topic 6

Processing topic 7/16: U.K. supported U.S. mission to seize Russian-flagged Bella-1 in the North Atlantica87.India plans to scrap curbs on Chinese firms bidding for government contracts
✓ Successfu

In [25]:
## Save all explanations to a text file

output_file = "output/Newsforkids_Explanations.txt"

# Ensure output directory exists
Path(output_file).parent.mkdir(parents=True, exist_ok=True)

# Write all explanations to the file
with open(output_file, 'w', encoding='utf-8') as f:
    f.write("=" * 80 + "\n")
    f.write("NEWS FOR KIDS - Kid-Friendly News Explanations\n")
    f.write(f"Generated on: {datetime.now().strftime('%B %d, %Y at %I:%M %p')}\n")
    f.write("=" * 80 + "\n\n")
    
    for explanation in all_explanations:
        f.write(explanation)
        f.write("-" * 80 + "\n\n")

print(f"✓ All explanations saved to: {output_file}")
print(f"Total topics processed: {len(all_explanations)}")

✓ All explanations saved to: output/Newsforkids_Explanations.txt
Total topics processed: 16


In [ ]:
## Display a preview of the saved content

print("\nPreview of saved content:\n")
with open(output_file, 'r', encoding='utf-8') as f:
    content = f.read()
    print(content[:1000])  # Show first 1000 characters
    if len(content) > 1000:
        print("\n... (content truncated for preview) ...")